In [1]:
from pathlib import Path
import hashlib
import json
import zipfile


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def inspect_checkpoint(path):
    path = Path(path)

    report = {
        "filename": path.name,
        "sha256": sha256_file(path),
    }

    with zipfile.ZipFile(path) as archive:
        names = set(archive.namelist())

        for filename in [
            "_stable_baselines3_version",
            "system_info.txt",
        ]:
            report[filename] = (
                archive.read(filename).decode(
                    "utf-8", errors="replace"
                ).strip()
                if filename in names
                else None
            )

        if "data" in names:
            data = json.loads(
                archive.read("data").decode("utf-8")
            )

            for key in [
                "seed",
                "num_timesteps",
                "n_envs",
                "n_steps",
                "batch_size",
                "n_epochs",
                "gamma",
            ]:
                value = data.get(key)
                if value is None or isinstance(
                    value, (str, int, float, bool)
                ):
                    report[key] = value
                else:
                    report[key] = (
                        "Stored in a non-scalar form; not deserialized"
                    )

    return report

# ระบุที่อยู่ไฟล์โมเดล .zip ของทั้ง 3 วิธี
checkpoints = {
    "Single Holdout": Path("Single Holdout/best_model.zip"),
    "Expanding Window": Path("Expanding window/best_model.zip"),
    "Sliding Window": Path("Sliding Window/best_model.zip"),
}

all_reports = {}

for method, ckpt_path in checkpoints.items():
    if ckpt_path.is_file():
        all_reports[method] = inspect_checkpoint(ckpt_path)
    else:
        print(f"[Warning] ไม่พบไฟล์ของ {method} ที่: {ckpt_path}")

# แสดงผลและบันทึกเป็นไฟล์ JSON รวมเพียงไฟล์เดียว
if all_reports:
    print(json.dumps(all_reports, ensure_ascii=False, indent=2))
    
    Path("all_checkpoints_inspection.json").write_text(
        json.dumps(all_reports, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("\nบันทึกผลลัพธ์ทั้งหมดลงในไฟล์ all_checkpoints_inspection.json เรียบร้อยแล้ว")

{
  "Single Holdout": {
    "filename": "best_model.zip",
    "sha256": "e0ae2b6847a8966df804f1c8412456fae0b30ecaa8d8d661c093339bfb70b9bc",
    "_stable_baselines3_version": "2.9.0",
    "system_info.txt": "- OS: Windows-10-10.0.26200-SP0 10.0.26200\n- Python: 3.10.20\n- Stable-Baselines3: 2.9.0\n- PyTorch: 2.13.0+cu132\n- GPU Enabled: True\n- Numpy: 2.0.1\n- Cloudpickle: 3.1.2\n- Gymnasium: 1.3.0",
    "seed": 43,
    "num_timesteps": 4000,
    "n_envs": 8,
    "n_steps": 1024,
    "batch_size": 256,
    "n_epochs": 10,
    "gamma": 0.99
  },
  "Expanding Window": {
    "filename": "best_model.zip",
    "sha256": "548e8c3615dcea80ac523ffb6941569902547fae2a779733171e8050b5d67504",
    "_stable_baselines3_version": "2.9.0",
    "system_info.txt": "- OS: Windows-10-10.0.26200-SP0 10.0.26200\n- Python: 3.10.20\n- Stable-Baselines3: 2.9.0\n- PyTorch: 2.13.0+cu132\n- GPU Enabled: True\n- Numpy: 2.0.1\n- Cloudpickle: 3.1.2\n- Gymnasium: 1.3.0",
    "seed": 47,
    "num_timesteps": 4000,
    

In [2]:
import json
import zipfile
from pathlib import Path

def inspect_training_state(zip_path):
    with zipfile.ZipFile(zip_path, "r") as archive:
        data = json.loads(
            archive.read("data").decode("utf-8")
        )

    keys = [
        "seed",
        "num_timesteps",
        "_n_updates",
        "n_envs",
        "n_steps",
        "n_epochs",
    ]

    return {key: data.get(key, None) for key in keys}

# ระบุตำแหน่งไฟล์ best_model.zip ของทั้ง 3 วิธี
checkpoints = {
    "Single Holdout": Path("Single Holdout/best_model.zip"),
    "Expanding Window": Path("Expanding window/best_model.zip"),
    "Sliding Window": Path("Sliding Window/best_model.zip"),
}

results = {}
for method, path in checkpoints.items():
    if path.is_file():
        results[method] = inspect_training_state(path)
    else:
        print(f"ไม่พบไฟล์ที่: {path}")

# แสดงผลลัพธ์บนหน้าจอ
print(json.dumps(results, ensure_ascii=False, indent=2))

{
  "Single Holdout": {
    "seed": 43,
    "num_timesteps": 4000,
    "_n_updates": 0,
    "n_envs": 8,
    "n_steps": 1024,
    "n_epochs": 10
  },
  "Expanding Window": {
    "seed": 47,
    "num_timesteps": 4000,
    "_n_updates": 0,
    "n_envs": 8,
    "n_steps": 1024,
    "n_epochs": 10
  },
  "Sliding Window": {
    "seed": 47,
    "num_timesteps": 4000,
    "_n_updates": 0,
    "n_envs": 8,
    "n_steps": 1024,
    "n_epochs": 10
  }
}
